# Stroke Risk — Exploratory Data Analysis

**Goal:** understand who is at risk of stroke and surface every data issue we must handle before modelling.

> **Leak-free rule:** this notebook looks at the **training split only**. The validation and test sets stay sealed so our later evaluation stays honest.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, gaussian_kde

from stroke_risk import data
from stroke_risk import plotting as pl

pl.set_theme()

# Explore the TRAINING split only.
splits = data.split_data(data.load_raw())
train = splits.train

CATEGORICAL = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
NUMERIC = ['age', 'avg_glucose_level', 'bmi']
train.shape

## 1. The dataset at a glance

In [ ]:
train.info()
train.head()

In [ ]:
# Numeric summary
train.describe().round(2)

In [ ]:
# Categorical summary: how many distinct values, and the most common one
train[CATEGORICAL].describe()

## 2. Stroke is rare

This class imbalance is the central challenge of the whole project — a model that predicts "no stroke" for everyone would already be ~95% accurate, yet useless.

In [ ]:
rate = train['stroke'].mean()

fig, ax = plt.subplots(figsize=(9, 1.9))
ax.barh(0, (1 - rate) * 100, color=pl.MUTED)
ax.barh(0, rate * 100, left=(1 - rate) * 100, color=pl.ACCENT)
ax.text((1 - rate) * 50, 0, f'No stroke   {100 * (1 - rate):.1f}%',
        ha='center', va='center', color='white', fontsize=11, fontweight='bold')
ax.text(100, 0.75, f'Stroke   {100 * rate:.1f}%',
        ha='right', va='bottom', color=pl.ACCENT, fontsize=11, fontweight='bold')
ax.set_xlim(0, 100)
ax.set_ylim(-0.6, 0.6)
ax.axis('off')
pl.add_titles(ax, 'Only about 1 in 20 patients had a stroke',
              'Share of patients by outcome (training set)')
plt.show()

## 3. Missing data

In [ ]:
miss = train.isna().mean().mul(100)
miss = miss[miss > 0].sort_values()

fig, ax = plt.subplots(figsize=(7, 2.2))
bars = ax.barh(miss.index, miss.values, color=pl.ACCENT)
ax.bar_label(bars, fmt='%.1f%%', padding=4, color=pl.SUBTLE, fontsize=10)
ax.set_xticks([])
ax.grid(False)
pl.despine(ax, left=False, bottom=True)
pl.add_titles(ax, 'Only BMI has missing values',
              'Share of missing entries per column')
plt.show()

### Is BMI missing *at random*?

If patients with a missing BMI have a different stroke rate, the missingness itself carries signal (missing **not** at random). That would be a reason to add a "BMI was missing" flag later.

In [ ]:
by_missing = train.assign(bmi_missing=train['bmi'].isna()).groupby('bmi_missing')['stroke'].agg(['mean', 'size'])
by_missing['mean'] = (by_missing['mean'] * 100).round(1)
by_missing.rename(columns={'mean': 'stroke_rate_%', 'size': 'patients'})

## 4. Numeric features vs stroke

Smoothed distributions (kernel density) let us compare the shape of each feature for patients who did and did not have a stroke.

In [ ]:
labels = {'age': 'Age', 'avg_glucose_level': 'Average glucose level', 'bmi': 'BMI'}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, col in zip(axes, NUMERIC):
    for cls in (1, 0):
        v = train.loc[train['stroke'] == cls, col].dropna()
        xs = np.linspace(v.min(), v.max(), 200)
        ys = gaussian_kde(v)(xs)
        ax.fill_between(xs, ys, color=pl.STROKE_COLORS[cls], alpha=0.55,
                        label=pl.STROKE_LABELS[cls])
        ax.plot(xs, ys, color=pl.STROKE_COLORS[cls], lw=1.5)
    ax.set_title(labels[col], loc='left', fontsize=12, fontweight='bold', color=pl.INK)
    ax.set_yticks([])
    ax.grid(False)
    pl.despine(ax, left=True)
axes[0].legend(loc='upper right')
fig.text(0, 1.02, 'Stroke patients skew toward higher age, glucose and BMI',
         fontsize=15, fontweight='bold', color=pl.INK)
plt.tight_layout()
plt.show()

## 5. How the numeric features relate

A correlation matrix shows which features move together and which are most linked to stroke.

In [ ]:
corr = train.select_dtypes('number').drop(columns='id').corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.mask(mask), cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        if not mask[i, j]:
            val = corr.iloc[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(val) > 0.55 else pl.INK)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.7, label='Pearson correlation')
pl.add_titles(ax, 'Age is the strongest numeric correlate of stroke',
              'Pearson correlation between numeric features')
plt.show()

## 6. Age — the dominant driver

In [ ]:
bins = [0, 20, 30, 40, 50, 60, 70, 80, 120]
names = ['<20', '20s', '30s', '40s', '50s', '60s', '70s', '80+']
band = pd.cut(train['age'], bins=bins, labels=names, right=False)
rate_by_age = train.groupby(band, observed=True)['stroke'].mean().mul(100)

fig, ax = plt.subplots()
bars = ax.bar(rate_by_age.index.astype(str), rate_by_age.values, color=pl.MUTED)
for b, name in zip(bars, rate_by_age.index):
    if name in ('70s', '80+'):
        b.set_color(pl.ACCENT)
ax.bar_label(bars, fmt='%.0f%%', padding=3, color=pl.SUBTLE, fontsize=9)
ax.set_yticks([])
ax.grid(False)
pl.despine(ax, left=True)
pl.add_titles(ax, 'Stroke risk climbs steeply with age',
              'Percentage of patients in each age group who had a stroke')
plt.show()

## 7. Every categorical feature, checked

Rather than eyeballing one chart per feature, we compute the stroke rate for **every level of every categorical feature** and rank them.

In [ ]:
rows = []
for col in CATEGORICAL:
    for level, sub in train.groupby(col):
        rows.append({'group': f'{col} = {level}', 'rate': sub['stroke'].mean() * 100, 'n': len(sub)})
ranked = pd.DataFrame(rows).sort_values('rate')

fig, ax = plt.subplots(figsize=(8.5, 7))
cut = ranked['rate'].median()
colors = [pl.ACCENT if r >= cut else pl.MUTED for r in ranked['rate']]
bars = ax.barh(ranked['group'], ranked['rate'], color=colors)
ax.bar_label(bars, labels=[f'{r:.1f}%  (n={n})' for r, n in zip(ranked['rate'], ranked['n'])],
             padding=4, color=pl.SUBTLE, fontsize=8)
ax.set_xticks([])
ax.set_xlim(0, ranked['rate'].max() * 1.25)
ax.grid(False)
pl.despine(ax, left=False, bottom=True)
pl.add_titles(ax, 'Which patient groups have the highest stroke rate?',
              'Stroke rate within each category level (training set)')
plt.show()

## 8. Comorbidities

`hypertension` and `heart_disease` are stored as 0/1, so we look at them separately.

In [ ]:
names, values, colors = [], [], []
for col, name in [('hypertension', 'Hypertension'), ('heart_disease', 'Heart disease')]:
    for present in (0, 1):
        names.append(f"{name}\n{'Yes' if present else 'No'}")
        values.append(train.loc[train[col] == present, 'stroke'].mean() * 100)
        colors.append(pl.ACCENT if present else pl.MUTED)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, values, color=colors)
ax.bar_label(bars, fmt='%.1f%%', padding=3, color=pl.SUBTLE, fontsize=10)
ax.set_yticks([])
ax.grid(False)
pl.despine(ax, left=True)
pl.add_titles(ax, 'Existing conditions multiply stroke risk',
              'Stroke rate with and without each condition')
plt.show()

## 9. Does BMI really differ by outcome?

The KDE hinted stroke patients have higher BMI. Because BMI is skewed (not normal), we test the difference with the non-parametric **Mann–Whitney U** test.

In [ ]:
pos = train.loc[train['stroke'] == 1, 'bmi'].dropna()
neg = train.loc[train['stroke'] == 0, 'bmi'].dropna()
u, p = mannwhitneyu(pos, neg, alternative='two-sided')
print(f'median BMI   stroke: {pos.median():.1f}   no stroke: {neg.median():.1f}')
print(f'Mann-Whitney U p-value: {p:.2e}')

**Leakage warning.** The test confirms BMI differs by outcome — so it is tempting to fill missing BMI with the median *of each stroke group*. **We must not do that:** the stroke label is the target, and using it to build a feature leaks the answer into the model. We will impute with the **overall training-set median** instead, and optionally add a "BMI was missing" flag.

## 10. Subgroups and quirks

In [ ]:
print('gender counts        :', dict(train['gender'].value_counts()))
print('work_type counts     :', dict(train['work_type'].value_counts()))
print('smoking_status counts:', dict(train['smoking_status'].value_counts()))
print('age  min / max       :', train['age'].min(), '/', train['age'].max())
print('child stroke cases   :', len(train[(train['work_type'] == 'children') & (train['stroke'] == 1)]))

In [ ]:
# The single rare-gender patient
train[train['gender'] == 'Other']

## 11. Feature-engineering ideas (for later)

Clinical reference ranges we could turn into features in a later phase:

- **Glucose flags** — elevated fasting glucose (diabetes risk) above clinical thresholds.
- **BMI category** — underweight / normal / overweight / obese bands.
- **`bmi_missing`** — an indicator column if the missingness turns out to be informative (see section 3).

## Findings → pipeline decisions

_Confirmed from the charts above:_

- **Target imbalance (~5%)** → stratified splits (done) + imbalance handling during modelling.
- **`bmi` missing (~4%)** → impute with **train-set median only** (never by the target — see section 9).
- **`id`** → drop (identifier, no predictive value).
- **Quirks:** one `gender = "Other"`, `age` stored as fractions, `smoking_status = "Unknown"` acts as its own category.
- **Strongest signals:** age (by far), glucose, BMI, hypertension, heart disease.
- **Candidate future features:** glucose/BMI clinical bands, `bmi_missing` flag.